# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates loading, inspecting, and performing exploratory data analysis (EDA) on the FAIR² dataset using the `mlcroissant` library and Pandas.

### Dataset Source
The dataset is described by a Croissant schema accessible via this URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}\n")
print(f"Authors: {getattr(metadata, 'author', None)}\n")

## 2. Data Overview
List available record sets and their properties using their `@id` identifiers.

In [ ]:
# List all record sets available in the Croissant schema
record_sets = list(dataset.record_sets)
if record_sets:
    print(f"Total record sets found: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"@id: {rs['@id']}")
        print(f"  name: {rs.get('name', 'N/A')}")
        print(f"  description: {rs.get('description', 'N/A')}")
        if 'field' in rs:
            print(f"  Fields:")
            for f in rs['field']:
                print(f"    @id: {f['@id']} | name: {f.get('name', 'N/A')}")
        print()
else:
    print("No record sets available in this dataset schema.")

## 3. Data Extraction
If record sets exist, extract their data as DataFrames using their `@id` fields. Otherwise, indicate no record sets.

In [ ]:
# Extract data for each record set
dataframes = dict()

if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
    for record_set_id in record_set_ids:
        print(f"Loading records for record set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print("  No records found in this record set.")
    print("\nDataFrames loaded for each record set, referenced by their @id.")
else:
    print("No record sets present to extract records from. Data extraction cannot proceed.")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA workflows such as filtering, normalization, and grouping, referencing all fields by their `@id`. If necessary, select fields/columns dynamically.

In [ ]:
# Example: Filter on a numeric field by @id and normalize

import numpy as np

if dataframes:
    # Select the first record set for EDA
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Exploring DataFrame from record set @id: {main_record_set_id}\n")

    # Identify possible numeric fields by @id
    numeric_candidates = df.select_dtypes(include=np.number).columns.tolist()
    print(f"Numeric fields (by @id): {numeric_candidates}")

    # If numeric fields exist, proceed
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]

        # Filter records where the value is above a threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical or key field (choose next available field)
        group_candidates = [col for col in df.columns if col != numeric_field_id]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable field found for grouping.")
    else:
        print("No numeric fields found in this DataFrame.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field or relationships between fields (by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram and scatter for numeric fields if available
if dataframes and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter vs the first group field if exists
    if group_field and group_field in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable fields available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and inspect a Croissant-described dataset using `mlcroissant`, referencing record sets, fields, and columns by their `@id`. We reviewed dataset metadata, record structure, performed basic EDA, and visualized selected features. Adapt the workflow for your analysis and downstream tasks!